# VERA v2 — M2 detector inference

Run this notebook after attaching the `vera-v2-inputs` bundle and the complete 448x448 image dataset. It uses the final YOLO checkpoint, shards inference across two T4 GPUs, aligns predictions to the authoritative M3 manifest, and prepares the separate Kaggle dataset `vera-v2-m2-detector-outputs`.

In [ ]:
from pathlib import Path
import sys, shutil, subprocess, zipfile
REPO_URL = 'https://github.com/hiennguyendang/phase_2_3_4_5.git'
def bootstrap_repo():
    target=Path('/kaggle/working/vera_repo')
    if (target/'.git').exists():
        try:
            subprocess.run(['git','-C',str(target),'fetch','origin','main'],check=True,stdout=subprocess.DEVNULL)
            subprocess.run(['git','-C',str(target),'reset','--hard','origin/main'],check=True,stdout=subprocess.DEVNULL)
            print('source: synced GitHub commit',subprocess.check_output(['git','-C',str(target),'rev-parse','HEAD'],text=True).strip())
            return target
        except Exception as exc: print('[refresh] existing clone unavailable:',exc)
    if (target/'phase_2/scripts/yolo/5-infer_yolo.py').exists(): return target
    if target.exists(): shutil.rmtree(target)
    try:
        subprocess.run(['git','clone',REPO_URL,str(target)],check=True)
        print('source: GitHub commit',subprocess.check_output(['git','-C',str(target),'rev-parse','HEAD'],text=True).strip())
        return target
    except Exception as exc: print('[fallback] GitHub clone unavailable:',exc)
    root=Path('/kaggle/input/datasets/nguynnghin/vera-v2-code'); archives=list(root.glob('*.zip')) if root.exists() else []
    candidates=[root]
    if len(archives)==1:
        extracted=Path('/kaggle/working/vera_v2_code_extracted'); extracted.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(archives[0]) as zf: zf.extractall(extracted)
        candidates=[extracted]+list(extracted.glob('*/'))
    for candidate in candidates:
        if (candidate/'phase_2/scripts/yolo/5-infer_yolo.py').exists(): shutil.copytree(candidate,target,dirs_exist_ok=True); print('source: Kaggle code dataset',candidate); return target
    raise RuntimeError('GitHub unavailable; attach /kaggle/input/datasets/nguynnghin/vera-v2-code')
REPO_DIR=bootstrap_repo()
sys.path.insert(0,str(REPO_DIR/'kaggle_notebooks'))

In [ ]:
%pip install -q ultralytics==8.4.95
import torch
print('python:', sys.version)
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'gpus:', torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count() >= 1, 'enable a Kaggle GPU accelerator'
for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i))

In [ ]:
import json, hashlib, shutil, zipfile
from vera_common import find_bundle, find_image_root, KAGGLE_DATASET_ROOT
bundle = find_bundle(minimal=True)
IMAGE_ROOT = find_image_root()
manifest_path = bundle / 'm3_labels_base' / 'manifest.jsonl'
weight = bundle / 'm2_detector' / 'last.pt'
assert IMAGE_ROOT.exists(), IMAGE_ROOT
assert manifest_path.exists() and weight.exists()
rows = [json.loads(x) for x in manifest_path.read_text(encoding='utf-8').splitlines() if x.strip()]
manifest_ids = {str(x['image_id']) for x in rows}
print('bundle:', bundle)
print('manifest rows/unique:', len(rows), len(manifest_ids))
print('image root exists; full image coverage is checked after inference/manifest alignment')
h = hashlib.sha256(weight.read_bytes()).hexdigest().upper()
print('YOLO SHA256:', h)
assert h == '71D4B4E3B173CC046FC45C7120B6CF4489C384CEAAEC9F08231182108A40DA56'

In [ ]:
# Deterministic, shard-level resume: completed shards are skipped; a partial shard is restarted fresh.
import os, subprocess, sys, shutil, time
from concurrent.futures import ThreadPoolExecutor
out = Path('/kaggle/working/vera_v2_detector_outputs')
shards_root = out / 'shards'; shards_root.mkdir(parents=True, exist_ok=True)
# If a previous partial notebook output is attached, restore only completed shards.
resume_roots = list(KAGGLE_DATASET_ROOT.glob('*/shards')) + list(KAGGLE_DATASET_ROOT.glob('*/*/shards'))
for archive in KAGGLE_DATASET_ROOT.glob('*/*partial*.zip'):
    extracted=Path('/kaggle/working/vera_m2_partial_inputs')/archive.stem; extracted.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as zf: zf.extractall(extracted)
    resume_roots += list(extracted.glob('**/shards'))
for src_root in resume_roots:
    for src in src_root.glob('pred_shard_*'):
        if (src/'_COMPLETE.json').exists(): shutil.copytree(src, shards_root/src.name, dirs_exist_ok=True)
script = REPO_DIR / 'phase_2/scripts/yolo/5-infer_yolo.py'
n_gpu = min(2, max(1, torch.cuda.device_count()))
print('using inference GPUs:', n_gpu)
RUN_BENCHMARK = True
BENCHMARK_IMAGES_PER_GPU = 128
MAX_ESTIMATED_HOURS = 8.5
SHARDS_TO_RUN = list(range(n_gpu))  # set [0] or [1] to split a long campaign across sessions
if RUN_BENCHMARK and not all((shards_root/f'pred_shard_{s}'/'_COMPLETE.json').exists() for s in SHARDS_TO_RUN):
    bench_root=Path('/kaggle/temp/vera_m2_benchmark'); bench_root.mkdir(parents=True,exist_ok=True)
    cache=out/'benchmark.json'
    if cache.exists():
        durations=json.loads(cache.read_text())['durations']; print('reusing cached benchmark:',durations)
    else:
        def benchmark_shard(shard):
            t0=time.perf_counter()
            subprocess.run([sys.executable,str(script),'--weights',str(weight),'--source',str(IMAGE_ROOT),'--manifest',str(manifest_path),
                            '--out',str(bench_root/f'shard_{shard}'),'--imgsz','1024','--conf','0.25','--iou','0.50',
                            '--batch','16','--device',str(shard),'--no-per-image','--shard-index',str(shard),'--num-shards',str(n_gpu),
                            '--limit',str(BENCHMARK_IMAGES_PER_GPU)],cwd=REPO_DIR,stdout=subprocess.DEVNULL,check=True)
            return time.perf_counter()-t0
        with ThreadPoolExecutor(max_workers=len(SHARDS_TO_RUN)) as pool:
            durations=list(pool.map(benchmark_shard,SHARDS_TO_RUN))
        cache.write_text(json.dumps({'durations':durations,'images_per_gpu':BENCHMARK_IMAGES_PER_GPU},indent=2))
    shard_images=(len(manifest_ids)+n_gpu-1)//n_gpu  # conservative lower bound; extra images are reported after merge
    eta_hours=max(durations)*shard_images/BENCHMARK_IMAGES_PER_GPU/3600
    print(f'benchmark: {BENCHMARK_IMAGES_PER_GPU} images/GPU; estimated selected-shard time: {eta_hours:.2f} h')
    if eta_hours > MAX_ESTIMATED_HOURS:
        raise RuntimeError(f'estimate exceeds {MAX_ESTIMATED_HOURS} h; set SHARDS_TO_RUN=[0] or [1], save the partial output, and run the other shard in a second session')
def run_shard(shard):
    shard_out = shards_root / f'pred_shard_{shard}'
    pred = shard_out/'predictions.jsonl'; marker = shard_out/'_COMPLETE.json'
    if pred.exists() and marker.exists():
        print(f'[GPU {shard}] resume: completed shard found, skipping')
        return
    shard_out.mkdir(parents=True, exist_ok=True); marker.unlink(missing_ok=True)
    cmd = [sys.executable, str(script), '--weights', str(weight), '--source', str(IMAGE_ROOT), '--manifest', str(manifest_path),
           '--out', str(shard_out), '--imgsz', '1024', '--conf', '0.25', '--iou', '0.50',
           '--batch', '16', '--device', str(shard), '--no-per-image', '--shard-index', str(shard), '--num-shards', str(n_gpu)]
    with (out/f'infer_shard_{shard}.log').open('a',encoding='utf-8') as log:
        p=subprocess.Popen(cmd,cwd=REPO_DIR,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush(); print(f'[GPU {shard}] {line}',end='')
        rc=p.wait()
    assert rc==0, f'shard {shard} failed with rc={rc}'
    count=sum(1 for _ in pred.open(encoding='utf-8'))
    marker.write_text(json.dumps({'shard':shard,'num_shards':n_gpu,'records':count,'completed_at':time.time()},indent=2))
with ThreadPoolExecutor(max_workers=len(SHARDS_TO_RUN)) as pool:
    list(pool.map(run_shard, SHARDS_TO_RUN))
complete_shards=[s for s in range(n_gpu) if (shards_root/f'pred_shard_{s}'/'_COMPLETE.json').exists()]
if len(complete_shards) < n_gpu:
    partial=Path('/kaggle/working/vera_v2_detector_partial'); partial.mkdir(parents=True,exist_ok=True)
    for s in complete_shards: shutil.copytree(shards_root/f'pred_shard_{s}',partial/'shards'/f'pred_shard_{s}',dirs_exist_ok=True)
    partial_zip=shutil.make_archive('/kaggle/working/vera-v2-m2-partial','zip',root_dir=partial)
    print(f'partial shards saved: {complete_shards}; upload {partial_zip} as a Kaggle dataset before the next session')
else:
    print('all inference shards completed:', n_gpu)

In [ ]:
# Merge, align to manifest rows, and validate coverage.
if len(complete_shards) < n_gpu:
    raise RuntimeError('partial M2 output is ready; upload it and rerun with the missing shard in SHARDS_TO_RUN')
from collections import Counter
preds = []
for shard in range(n_gpu):
    p = shards_root / f'pred_shard_{shard}' / 'predictions.jsonl'
    preds.extend(json.loads(x) for x in p.read_text(encoding='utf-8').splitlines() if x.strip())
ids = [str(x['image_id']) for x in preds]
dupes = [k for k,v in Counter(ids).items() if v != 1]
assert not dupes, f'duplicate/missing shard ownership for IDs: {dupes[:5]}'
assert not (manifest_ids - set(ids)), (len(set(ids)), len(manifest_ids), len(manifest_ids-set(ids)))
print('extra predictions outside manifest:', len(set(ids) - manifest_ids))
merged = out / 'predictions.jsonl'
merged.write_text(''.join(json.dumps(x, separators=(',', ':'))+'\n' for x in sorted(preds, key=lambda z: str(z['image_id']))), encoding='utf-8')
aligned = out / 'm3_labels_detector_v2'
aligned.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, str(REPO_DIR / 'phase_3/scripts/3-boxes_from_pred.py'),
                '--pred', str(merged), '--manifest', str(manifest_path), '--out-dir', str(aligned), '--input-res', '448'],
               cwd=REPO_DIR, check=True)
import numpy as np
boxes = np.load(aligned / 'boxes_det.npy', mmap_mode='r')
present = np.load(aligned / 'present_mask_det.npy', mmap_mode='r')
print('boxes:', boxes.shape, 'present:', present.shape)
assert boxes.shape == (len(rows), 29, 4) and present.shape == (len(rows), 29)
assert np.isfinite(boxes).all() and ((boxes >= 0) & (boxes <= 448)).all()

In [ ]:
from vera_common import write_json
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(8<<20), b''): h.update(b)
    return h.hexdigest()
provenance = {
  'detector_checkpoint_sha256': sha256(weight),
  'prediction_jsonl_sha256': sha256(merged),
  'imgsz': 1024, 'conf': 0.25, 'iou': 0.50, 'batch': 16,
  'input_coordinate_system': 'normalized YOLO corners converted to 448x448',
  'manifest': str(manifest_path), 'manifest_rows': len(rows),
  'unique_manifest_images': len(manifest_ids),
  'boxes_file': 'boxes_det.npy', 'present_mask_file': 'present_mask_det.npy',
  'shape': list(present.shape), 'mean_regions_per_image': float(present.sum(axis=1).mean()),
  'ultralytics_version': '8.4.95', 'num_shards': n_gpu
}
write_json(aligned / 'detector_provenance.json', provenance)
export = Path('/kaggle/working/vera_v2_detector_kaggle_dataset')
export.mkdir(parents=True,exist_ok=True)
shutil.copy2(merged,export/'predictions.jsonl')
shutil.copytree(aligned,export/'m3_labels_detector_v2',dirs_exist_ok=True)
for log in out.glob('*.log'): shutil.copy2(log,export/log.name)
write_json(export/'_SUCCESS.json', {'status':'complete','records':len(preds),'manifest_rows':len(rows)})
# Convenience archive contains only final artifacts, not duplicate shard JSONLs.
archive = shutil.make_archive('/kaggle/working/vera-v2-m2-detector-output','zip',root_dir=export)
print('M2 complete. Create the Kaggle dataset vera-v2-m2-detector-outputs from:', export)
print('or upload and attach this zip (the downstream notebook can extract it):', archive)